In [1]:
import pandas as pd
from pathlib import Path
from functools import reduce
import re

KEYS = ["PAIS", "date", "Scenario_name"]
PAIS     = "CO"
V_CALIB  = "01mar26"

# Mapea abreviaturas que puedan aparecer en el nombre del archivo al nombre
# "oficial" del producto (para que la columna final quede consistente,
# sin importar si el archivo se llama Impacto_cons_CO o Impacto_consumo_CO)
ALIAS_PRODUCTO = {
    "cons": "consumo",
    "consumo": "consumo",
    "corporativo": "corporativo",
    "corp": "corporativo",
    "pyme": "pymes",
    "pymes": "pymes",
    "tarjeta": "tarjeta",
    "vehiculo": "vehiculo",
    "vivi": "vivienda",
    "vivienda": "vivienda",
}

pa_dir = Path.cwd()

archivos = sorted(pa_dir.glob(f"Impacto_*_{PAIS}*.csv"))

if not archivos:
    raise FileNotFoundError(f"No se encontraron archivos Impacto_*_{PAIS}.csv en {pa_dir}")

frames = []
for f in archivos:
    df_tmp = pd.read_csv(f)

    # Detectar el nombre del producto a partir del NOMBRE DEL ARCHIVO
    # (no de la columna interna, que puede venir mal etiquetada -- como pasa
    # con Impacto_tarjeta_CO.csv, que trae la columna "Impacto_consumo" por error)
    match = re.search(r"Impacto_([a-zA-Z]+)", f.name)
    if not match:
        raise ValueError(f"No se pudo detectar el producto en el nombre: {f.name}")
    producto_raw = match.group(1).lower()
    producto = ALIAS_PRODUCTO.get(producto_raw, producto_raw)

    # Encontrar la columna de impacto real dentro del archivo
    col_impacto = [c for c in df_tmp.columns if c.startswith("Impacto_")]
    if len(col_impacto) != 1:
        raise ValueError(f"{f.name}: se esperaba 1 columna Impacto_*, se encontraron {col_impacto}")

    nuevo_nombre = f"Impacto_{producto}"
    if col_impacto[0] != nuevo_nombre:
        print(f"Aviso: {f.name} traia la columna '{col_impacto[0]}', renombrada a '{nuevo_nombre}'")
    df_tmp = df_tmp.rename(columns={col_impacto[0]: nuevo_nombre})

    frames.append(df_tmp)

df = reduce(lambda left, right: pd.merge(left, right, on=KEYS, how="outer"), frames)

df["date"] = pd.to_datetime(df["date"], format="%d%b%Y")
df = df.sort_values(KEYS).reset_index(drop=True)

salida = pa_dir / f"fwl_{PAIS}_{V_CALIB}.csv"
df.to_csv(salida, index=False)

print(f"Archivos procesados: {[f.name for f in archivos]}")
print(f"Dimensiones resultado: {df.shape[0]} filas x {df.shape[1]} columnas")
print(f"Columnas: {list(df.columns)}")
print(f"Guardado en: {salida}")

Aviso: Impacto_tarjeta_CO.csv traia la columna 'Impacto_consumo', renombrada a 'Impacto_tarjeta'
Archivos procesados: ['Impacto_cons_CO (1).csv', 'Impacto_corporativo_CO (1) (1).csv', 'Impacto_pyme_CO.csv', 'Impacto_tarjeta_CO.csv', 'Impacto_vivi_CO (1).csv']
Dimensiones resultado: 3576 filas x 8 columnas
Columnas: ['PAIS', 'date', 'Scenario_name', 'Impacto_consumo', 'Impacto_corporativo', 'Impacto_pymes', 'Impacto_tarjeta', 'Impacto_vivienda']
Guardado en: /content/fwl_CO_01mar26.csv
